# Exercises

There are three exercises in this notebook:

1. Use the cross-validation method to test the linear regression with different $\alpha$ values, at least three.
2. Implement a SGD method that will train the Lasso regression for 10 epochs.
3. Extend the Fisher's classifier to work with two features. Use the class as the $y$.

## 1. Cross-validation linear regression

You need to change the variable ``alpha`` to be a list of alphas. Next do a loop and finally compare the results.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = (np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1).reshape(15, 1))
y = (np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1).reshape(15, 1))

x = np.asmatrix(np.c_[np.ones((15, 1)), x])

I = np.identity(2)

alphas = [0.001, 0.01, 0.1, 1.0, 10.0]  # change here

# add 1-3 line of code here
for alpha in alphas:
    w = np.linalg.inv(x.T * x + alpha * I) * x.T * y
    w = w.ravel()

    # add 1-3 lines to compare the results
    y_pred = np.asarray(x * w.T)
    mse = np.mean(np.asarray(y - y_pred) ** 2)
    print(f"alpha={alpha:<6} | w={np.asarray(w).flatten()} | MSE={mse:.4f}")

: 

## 2. Implement based on the Ridge regression example, the Lasso regression.

Please implement the SGD method and compare the results with the sklearn Lasso regression results. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso


def sgd(x, y, alpha, lr=1e-6, epochs=10):
    # your code goes here
    n_samples, n_features = x.shape
    w = np.zeros(n_features)

    for epoch in range(epochs):
        for i in range(n_samples):
            xi = np.asarray(x[i]).flatten()
            yi = y[i, 0]
            prediction = np.dot(xi, w)
            error = prediction - yi

            gradient = xi * error + alpha * np.sign(w)
            w = w - lr * gradient

        y_pred = np.asarray(x @ np.matrix(w).T)
        mse = np.mean(np.asarray(y - y_pred) ** 2)
        print(f"  Epoch {epoch+1}/{epochs} | w={w} | MSE={mse:.4f}")

    return w


x = (np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1).reshape(15, 1))
y = (np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1).reshape(15, 1))

x = np.asmatrix(np.c_[np.ones((15, 1)), x])

I = np.identity(2)
alpha = 0.1

w = sgd(x, y, alpha, lr=1e-6, epochs=10)
w = np.asmatrix(w).T

print("SGD Lasso weights:", np.asarray(w).flatten())


heights_raw = np.array([188, 181, 197, 168, 167, 187, 178, 194, 140, 176, 168, 192, 173, 142, 176]).reshape(-1, 1)
weights_raw = np.array([141, 106, 149, 59, 79, 136, 65, 136, 52, 87, 115, 140, 82, 69, 121]).reshape(-1, 1)

lasso_sklearn = Lasso(alpha=alpha)
lasso_sklearn.fit(heights_raw, weights_raw)
print(
    "sklearn Lasso:     intercept={:.4f}, coef={:.4f}".format(
        lasso_sklearn.intercept_[0], lasso_sklearn.coef_[0]
    )
)

## 3. Extend the Fisher's classifier

Write numpy code that performs Fisher classification using all the features of Iris data. Choose species nr 1 and 2 as two classes. Plot original data and their projection. Using obtained model classify the flower with the folowing features: [6.45, 2.85, 4.25, 1.25].

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn import preprocessing

iris_data, iris_labels = load_iris(return_X_y=True)
iris_data = np.array(
    preprocessing.normalize(iris_data)
)

x1 = iris_data[np.where(iris_labels == 1)]
x2 = iris_data[np.where(iris_labels == 2)]

mean_x1, mean_x2 = np.mean(x1, axis=0), np.mean(x2, axis=0)

Sw = np.dot((x1 - mean_x1).T, (x1 - mean_x1)) + np.dot((x2 - mean_x2).T, (x2 - mean_x2))

w = np.dot(np.linalg.inv(Sw), (mean_x2 - mean_x1))  ## weights

proj_x1 = np.dot(x1, w)
proj_x2 = np.dot(x2, w)

# Plot original data (first two features) and their projection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x1[:, 0], x1[:, 1], "bo", label="Class 1 (versicolor)")
axes[0].plot(x2[:, 0], x2[:, 1], "go", label="Class 2 (virginica)")
axes[0].set_xlabel("Feature 0 (sepal length, normalized)")
axes[0].set_ylabel("Feature 1 (sepal width, normalized)")
axes[0].set_title("Original data (first two features)")
axes[0].legend()

axes[1].plot(proj_x1, [0] * x1.shape[0], "bo", label="Class 1 (versicolor)")
axes[1].plot(proj_x2, [0] * x2.shape[0], "go", label="Class 2 (virginica)")
axes[1].set_xlabel("Projection onto Fisher direction")
axes[1].set_title("Fisher projection (1D)")
axes[1].legend()

plt.tight_layout()
plt.show()

new_sample = np.array([6.45, 2.85, 4.25, 1.25])
new_sample_normalized = new_sample / np.linalg.norm(new_sample)

projection = np.dot(new_sample_normalized, w)

threshold = (np.mean(proj_x1) + np.mean(proj_x2)) / 2

predicted_class = 1 if projection < threshold else 2
print(f"Projection of new sample: {projection:.4f}")
print(f"Threshold (midpoint of class means): {threshold:.4f}")
print(
    f"Predicted class: {predicted_class} ({'versicolor' if predicted_class == 1 else 'virginica'})"
)